# Login runner & certificate cache checks — M1-1, M1-2, M1-3, M1-5, M1-6, M1-7, M1-8, M1-9, M1-10, M1-11, M1-12, M1-14, M1-15, M1-38, M2-4, M3-4

Extended 2026-08-15 (Batch 6) with 5 more cases from the same theme: M1-8/M1-9 (the certificate-fetch step's own malformed-response handling), M1-7/M1-11/M1-38 (Flow 8/9's Search ABHA Account step -- non-JSON body, a single object instead of a list, and a missing txnId).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M1-10 / M1-12 — malformed `verify_otp` responses must not crash or misreport success

Both fixed in `tools/m1_test_suite/login_runner.py`'s `verify_login_otp()` — the one shared parsing point
7 of the 8 M1 login variants flow through.

- **M1-10:** a `"tokens"` section that's PRESENT but EMPTY (`{}`) used to be silently treated the same as a
  missing token, while the CLI still printed "Login succeeded." — misreporting a login that produced no
  usable token as a success.
- **M1-12:** an `"accounts"`/`"users"` field that comes back as something other than a list (e.g. a plain
  string) used to be passed straight through to `print_accounts()`/`select_account()`, which iterate it
  expecting dicts — for a string that means iterating individual characters and crashing with an
  `AttributeError` the first time `.get(...)` is called on one.

**Test approach:** `prompt()` (would otherwise block on real keyboard input) and `verify_otp()` (would
otherwise make a real ABDM network call) are both stubbed; `verify_login_otp()` itself is called directly,
unmodified.

**Pass criteria:** the empty-tokens case reports failure, not success; the non-list-accounts case doesn't
crash and is treated as zero accounts; a normal, well-formed response still works exactly as before.

In [2]:
from unittest.mock import patch

import tools.m1_test_suite.login_runner as login_runner

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"users": [{"name": "Test Patient"}], "tokens": {}})):
    result_10 = login_runner.verify_login_otp(action="phr/web/login/abha", scope=["abha-login"], txn_id="txn-1")

harness.check("empty-but-present 'tokens' section is reported as a failure, not a false 'Login succeeded'", result_10["x_token"] is None)

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"accounts": "not actually a list", "token": "real-token-xyz"})):
    try:
        result_12 = login_runner.verify_login_otp(action="profile/login", scope=["abha-login"], txn_id="txn-2")
        crashed = False
    except Exception:
        crashed = True
        result_12 = None

harness.check("non-list 'accounts' field does not crash the CLI", not crashed)
harness.check("non-list 'accounts' is treated as zero accounts, not passed through raw", result_12 is not None and result_12["accounts"] == [])

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"accounts": [{"name": "Real Patient"}], "token": "real-token-abc"})):
    result_normal = login_runner.verify_login_otp(action="profile/login", scope=["abha-login"], txn_id="txn-3")

harness.check("a normal, well-formed response still reports success with the real token/accounts", result_normal["x_token"] == "real-token-abc" and len(result_normal["accounts"]) == 1)


      Verifying OTP...

[FAIL] OTP verification returned a 200/success response but no usable token was found under either 'token' or 'tokens.token' -- treating this as a failure rather than reporting a successful login with nothing to show for it.
PASS -- empty-but-present 'tokens' section is reported as a failure, not a false 'Login succeeded'
      Verifying OTP...

[FAIL] OTP verification returned an 'accounts'/'users' field that isn't a list (got str: 'not actually a list') -- treating this as no accounts returned rather than crashing on it.

[OK] Login succeeded.
      No accounts were returned.
PASS -- non-list 'accounts' field does not crash the CLI
PASS -- non-list 'accounts' is treated as zero accounts, not passed through raw
      Verifying OTP...

[OK] Login succeeded.
      1 account(s) returned:
        [1] Real Patient  |  ABHA Number: None  |  ABHA Address: None
PASS -- a normal, well-formed response still reports success with the real token/accounts


True

---
## M1-1 — the ABDM public certificate cache self-heals within a bounded TTL

**Real-world scenario:** if ABDM ever rotates the RSA key behind their public certificate endpoint, a
long-running server process that cached the OLD certificate would keep using it forever (no restart, no
local signal that a rotation happened — the failure shows up as a generic ABDM-side decrypt error, not
anything distinguishable here). The fix bounds how long the cache can go stale: after
`_CERTIFICATE_TTL_SECONDS` (24h), the next call forces a fresh download.

**Test approach:** rather than actually waiting 24 hours (or temporarily shrinking the real TTL constant
and restarting the server, which is what the manual runbook version of this test requires), this notebook
directly ages the module's own cached timestamp past the TTL window — exercising the exact same `is_stale`
check `get_public_certificate()` runs, without waiting or touching the real constant.

**Pass criteria:** the first call downloads; an immediate second call reuses the cache (no re-download);
a call made after the TTL window has elapsed triggers a fresh download.

In [3]:
import time

import server.crypto as crypto

crypto.clear_certificate_cache()

download_calls = []
def fake_download():
    download_calls.append(time.monotonic())
    return object()

with patch.object(crypto, "download_public_certificate", fake_download):
    crypto.get_public_certificate()
    harness.check("first call downloads (cache was empty)", len(download_calls) == 1)

    crypto.get_public_certificate()
    harness.check("second call right after reuses the cache (no re-download)", len(download_calls) == 1)

    # Age the cached timestamp past the TTL window without actually waiting.
    crypto._cached_at = time.monotonic() - crypto._CERTIFICATE_TTL_SECONDS - 1

    crypto.get_public_certificate()
    harness.check("a call after the TTL window elapses triggers a fresh download (self-heals without a restart)", len(download_calls) == 2)

crypto.clear_certificate_cache()  # leave the module in a clean state for anything run after this


2026-08-15 19:14:33  -> ABDM public certificate downloaded
2026-08-15 19:14:33  -> ABDM public certificate refreshed (TTL expired)


PASS -- first call downloads (cache was empty)
PASS -- second call right after reuses the cache (no re-download)
PASS -- a call after the TTL window elapses triggers a fresh download (self-heals without a restart)


---
## M1-2 — Concurrent logins must not download the ABDM public certificate twice

**Real code under test:** `server/crypto.py` — `get_public_certificate()`.

**Real-world scenario:** two patients complete ABHA login at (almost) the exact same moment on a busy
morning. Both requests hit the certificate cache while it's empty (server just started) or just went stale.
Before this fix, both could see "no cached certificate" and both call `download_public_certificate()` --
wasteful at best, and at worst one caller's own `return _cached_certificate` could read back a DIFFERENT
certificate object than the one it itself just downloaded (whichever write landed last wins), which is a
genuinely confusing bug to chase down later.

**Fix:** a `threading.Lock` around the whole "check staleness, maybe download, write cache" sequence
serializes concurrent callers within this one process -- same pattern as `json_file_store.py`'s per-file
lock (M2-1) and M1-3's token-cache lock below.

**Pass criteria:** 10 concurrent callers racing an artificially slow download → exactly 1 real download,
and every caller gets back the identical certificate object.

In [4]:
import threading
import time
from unittest.mock import patch

import server.crypto as crypto

crypto.clear_certificate_cache()

download_calls = []
class FakeCert:
    def __init__(self, n):
        self.n = n

def slow_download():
    time.sleep(0.1)
    download_calls.append(time.monotonic())
    return FakeCert(len(download_calls))

with patch.object(crypto, "download_public_certificate", slow_download):
    results = []
    def worker():
        results.append(crypto.get_public_certificate())
    threads = [threading.Thread(target=worker) for _ in range(10)]
    [t.start() for t in threads]
    [t.join() for t in threads]

harness.check("10 concurrent callers -> exactly 1 real download (lock serialized them)", len(download_calls) == 1)
harness.check("every caller got back the SAME certificate object (no mismatched-write race)", len(set(id(r) for r in results)) == 1)

crypto.clear_certificate_cache()


2026-08-15 19:14:34  -> ABDM public certificate downloaded


PASS -- 10 concurrent callers -> exactly 1 real download (lock serialized them)
PASS -- every caller got back the SAME certificate object (no mismatched-write race)


---
## M1-3 — Concurrent logins must not scramble the cached ABDM gateway token

**Real code under test:** `server/utils.py` — `get_gateway_token()`.

**Real-world scenario:** same busy-morning setup as M1-2, but for the shared ABDM gateway access token
instead of the certificate. Before this fix, `_token_cache["access_token"]` and
`_token_cache["expires_at"]` were written as two separate statements with no lock -- two concurrent
refreshes could interleave their writes and leave the cache holding one refresh's token paired with a
*different* refresh's expiry, a genuinely mismatched pair that doesn't correspond to either real token
response ABDM actually issued.

**Fix:** same `threading.Lock` pattern as M1-2, wrapped around the whole check-and-maybe-refresh sequence.

**Pass criteria:** 10 concurrent callers racing an artificially slow token refresh → exactly 1 real refresh
call, every caller gets back the identical token string, and the cached expiry is set (the pair landed
together, not split across two different refreshes).

In [5]:
import threading
import time
from unittest.mock import patch

import server.utils as utils

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None

class FakeTokenResponse:
    def __init__(self, body):
        self._body = body
    def json(self):
        return self._body
    def raise_for_status(self):
        pass

gen_calls = []
def slow_generate_gateway_token():
    time.sleep(0.1)
    idx = len(gen_calls)
    gen_calls.append(idx)
    return FakeTokenResponse({"accessToken": f"TOKEN_{idx}", "expiresIn": 1800})

with patch("server.auth.generate_gateway_token", slow_generate_gateway_token):
    results = []
    def worker():
        results.append(utils.get_gateway_token())
    threads = [threading.Thread(target=worker) for _ in range(10)]
    [t.start() for t in threads]
    [t.join() for t in threads]

harness.check("10 concurrent callers -> exactly 1 real token refresh (lock serialized them)", len(gen_calls) == 1)
harness.check("every caller got back the SAME token string (no mismatched token/expiry write race)", len(set(results)) == 1)
harness.check("cached expires_at is set (paired write completed atomically under the lock)", utils._token_cache["expires_at"] is not None)

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None


2026-08-15 19:14:36  -> Gateway token refreshed


PASS -- 10 concurrent callers -> exactly 1 real token refresh (lock serialized them)
PASS -- every caller got back the SAME token string (no mismatched token/expiry write race)
PASS -- cached expires_at is set (paired write completed atomically under the lock)


---
## M1-5 — a broken/non-JSON login response (HTTP 200, unparseable body) must not crash the CLI

**Real-world scenario**: ABDM's login server (or something in front of it — a proxy, load balancer, misconfigured gateway) returns an HTTP 200 whose body isn't valid JSON at all (an HTML error page, truncated output, etc.). The old code called `response.json()` directly with no guard the moment `status_code == 200`, assuming a 200 always means a parseable body — an uncaught `ValueError` would crash the CLI mid-flow.

**Fix** (`tools/m1_test_suite/login_runner.py`, both `request_login_otp()` and `verify_login_otp()`): `response.json()` is now wrapped in try/except `ValueError`, reporting a clear failure and returning the function's normal "failed" contract instead of crashing — same shape as the M1-10/M1-12 guards already in this notebook.

In [6]:
from unittest.mock import patch

import tools.m1_test_suite.login_runner as login_runner

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "request_otp", lambda **kw: harness.FakeResponse(200, not_json=True)):
    try:
        result_req = login_runner.request_login_otp(action="profile/login", scope=["abha-login"], login_hint="mobile", login_id="enc", otp_system="abdm")
        crashed_req = False
    except Exception as exc:
        crashed_req = True
        result_req = None

harness.check("request_login_otp(): a 200 with a non-JSON body does NOT crash the CLI", not crashed_req)
harness.check("request_login_otp(): treated as a failed OTP request (returns None)", result_req is None)

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, not_json=True)):
    try:
        result_verify = login_runner.verify_login_otp(action="profile/login", scope=["abha-login"], txn_id="txn-m1-5")
        crashed_verify = False
    except Exception as exc:
        crashed_verify = True
        result_verify = None

harness.check("verify_login_otp(): a 200 with a non-JSON body does NOT crash the CLI", not crashed_verify)
harness.check("verify_login_otp(): treated as a failed verification (x_token is None)", result_verify is not None and result_verify["x_token"] is None)


      Requesting OTP...

[FAIL] OTP request returned a 200 status but the response body wasn't valid JSON (No JSON object could be decoded (simulated malformed body)) -- treating this as a failed request rather than crashing.
PASS -- request_login_otp(): a 200 with a non-JSON body does NOT crash the CLI
PASS -- request_login_otp(): treated as a failed OTP request (returns None)
      Verifying OTP...

[FAIL] OTP verification returned a 200 status but the response body wasn't valid JSON (No JSON object could be decoded (simulated malformed body)) -- treating this as a failed verification rather than crashing.
PASS -- verify_login_otp(): a 200 with a non-JSON body does NOT crash the CLI
PASS -- verify_login_otp(): treated as a failed verification (x_token is None)


True

---
## M1-6 — an unexpected token-expiry value from ABDM must not crash the gateway-token refresh

**Real-world scenario**: ABDM's `/sessions` response is missing `expiresIn` entirely (already handled — raises a caught `KeyError`), or carries it as a value that isn't a usable positive number: `null`, a string like `"1800"`, a negative number, or a boolean. The old code passed it straight into `timedelta(seconds=data["expiresIn"])` unchecked — `None`/a string raise an uncaught `TypeError` (the old `except (RequestException, KeyError, ValueError)` didn't catch that), and a negative value would silently mark the freshly-"refreshed" token as already-expired without ever raising anything.

**Fix** (`server/utils.py`, `get_gateway_token()`): `expiresIn` is now explicitly validated (must be a real positive number, not a bool) before being used, raising a clear `ValueError` — caught by the now-widened `except (KeyError, ValueError, TypeError)` — instead of an unhandled crash or a silently-broken cache.

In [7]:
import server.utils as utils

for bad_value, label in [(None, "null"), ("1800", "string"), (-5, "negative"), (True, "boolean")]:
    utils._token_cache["access_token"] = None
    utils._token_cache["expires_at"] = None

    def bad_generate_gateway_token(bv=bad_value):
        return harness.FakeResponse(200, {"accessToken": "TOKEN", "expiresIn": bv})

    with patch("server.auth.generate_gateway_token", bad_generate_gateway_token):
        try:
            utils.get_gateway_token()
            raised_clean = False
        except (KeyError, ValueError, TypeError):
            raised_clean = True
        except Exception:
            raised_clean = False

    harness.check(f"expiresIn={label} ({bad_value!r}) -> raised a clear, caught error (not an unhandled crash)", raised_clean)
    harness.check(f"expiresIn={label}: cache left clean, not a broken half-write", utils._token_cache["access_token"] is None)

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None


2026-08-15 19:14:42     [ERROR] Gateway token response was malformed: 'expiresIn' must be a positive number, got NoneType: None
2026-08-15 19:14:42     [ERROR] Gateway token response was malformed: 'expiresIn' must be a positive number, got str: '1800'
2026-08-15 19:14:42     [ERROR] Gateway token response was malformed: 'expiresIn' must be a positive number, got int: -5
2026-08-15 19:14:42     [ERROR] Gateway token response was malformed: 'expiresIn' must be a positive number, got bool: True


PASS -- expiresIn=null (None) -> raised a clear, caught error (not an unhandled crash)
PASS -- expiresIn=null: cache left clean, not a broken half-write
PASS -- expiresIn=string ('1800') -> raised a clear, caught error (not an unhandled crash)
PASS -- expiresIn=string: cache left clean, not a broken half-write
PASS -- expiresIn=negative (-5) -> raised a clear, caught error (not an unhandled crash)
PASS -- expiresIn=negative: cache left clean, not a broken half-write
PASS -- expiresIn=boolean (True) -> raised a clear, caught error (not an unhandled crash)
PASS -- expiresIn=boolean: cache left clean, not a broken half-write


---
## M1-14 — Flow 3 (Login using Mobile Number) missing its security token must not report a false success

**Real-world scenario**: Flow 3 is a two-step login — the OTP-verify step returns a short-lived T-Token, then `verify_user()` (account selection) returns the real X-Token. This case covers BOTH steps' "token missing" handling.

The T-Token step already goes through `verify_login_otp()`'s generic `token is None` guard (fixed for M1-10, above) — so a missing T-Token was already caught before this pass. What was NOT guarded: the second, separate `verify_user()` step has its own `token` field for the real X-Token, and the old `flows/login_mobile.py` printed "Login complete" unconditionally even if that came back `None` — reporting success for a login with no usable session token.

**Fix** (`tools/m1_test_suite/flows/login_mobile.py`): the `verify_user()` response's `token` is now checked explicitly, reporting a clear failure instead of a false "Login complete" when it's missing.

In [8]:
import tools.m1_test_suite.flows.login_mobile as login_mobile

# Case A: the T-Token step itself is missing its token -- exercised through the
# REAL verify_login_otp() (not mocked away), only faking the underlying
# verify_otp() HTTP call and the OTP prompt, to confirm the generic M1-10 guard
# inside verify_login_otp() really does cover this flow too, not just the 7
# non-mobile variants.
with patch.object(login_mobile, "request_login_otp", lambda **kw: "txn-m1-14a"), \
     patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"accounts": [{"ABHANumber": "AB-1"}]})), \
     patch.object(login_mobile, "prompt", lambda *a, **kw: "9999999999"), \
     patch.object(login_mobile, "encrypt", lambda v: v):
    result_a = login_mobile.run()

harness.check("Case A (T-Token step missing token): login_mobile.run() reports failure, not a false success", result_a["x_token"] is None)

# Case B: the T-Token step succeeds, but the LATER verify_user() step's response
# is missing 'token' -- the gap this batch actually fixed.
with patch.object(login_mobile, "request_login_otp", lambda **kw: "txn-m1-14b"), \
     patch.object(login_mobile, "verify_login_otp", lambda **kw: {"x_token": "T-TOKEN-abc", "accounts": [{"ABHANumber": "AB-1"}], "txn_id": "txn-m1-14b"}), \
     patch.object(login_mobile, "select_account", lambda accounts: accounts[0]), \
     patch.object(login_mobile, "verify_user", lambda **kw: harness.FakeResponse(200, {"someOtherField": "x"})), \
     patch.object(login_mobile, "prompt", lambda *a, **kw: "9999999999"), \
     patch.object(login_mobile, "encrypt", lambda v: v):
    result_b = login_mobile.run()

harness.check("Case B (verify_user() success but missing 'token'): reports failure, not a false 'Login complete'", result_b["x_token"] is None)



Flow 3: Login using Mobile Number
      Verifying OTP...

[FAIL] OTP verification returned a 200/success response but no usable token was found under either 'token' or 'tokens.token' -- treating this as a failure rather than reporting a successful login with nothing to show for it.
PASS -- Case A (T-Token step missing token): login_mobile.run() reports failure, not a false success

Flow 3: Login using Mobile Number
      Selecting account: ABHA Number AB-1...

[FAIL] Account selection (verify_user) returned a 200/success response but no usable token was found under 'token' -- treating this as a failed login rather than reporting success with nothing to show for it.
PASS -- Case B (verify_user() success but missing 'token'): reports failure, not a false 'Login complete'


True

---
## M1-15 — enrollment OTP-request response missing `txnId` must not silently proceed

**Real-world scenario**: Flow 1 (ABHA Enrollment via Aadhaar) requests an OTP, then submits it via `enroll_by_aadhaar(txn_id=...)`. The old code took `otp_body.get("txnId")` on faith — if ABDM's OTP-request response is missing `txnId`, `txn_id` silently becomes `None`, the user is still prompted for an OTP they can't meaningfully verify against anything, and `enroll_by_aadhaar()` would be called with a literal `txnId: null` sent to ABDM instead of failing fast locally.

**Fix** (`tools/m1_test_suite/flows/enrollment.py`): a missing/falsy `txnId` is now caught immediately after the OTP request, before the OTP prompt even happens — reports a clear failure and returns, `enroll_by_aadhaar()` is never called.

In [9]:
import tools.m1_test_suite.flows.enrollment as enrollment

def _boom(**kw):
    raise AssertionError("enroll_by_aadhaar() should NEVER be called when txnId was missing")

with patch.object(enrollment, "request_otp", lambda **kw: harness.FakeResponse(200, {"message": "OTP sent"})), \
     patch.object(enrollment, "enroll_by_aadhaar", _boom), \
     patch.object(enrollment, "prompt", lambda label: "999999999999" if "Aadhaar" in label else "9999999999"), \
     patch.object(enrollment, "encrypt", lambda v: v):
    try:
        result = enrollment.run()
        proceeded_anyway = False
    except AssertionError:
        proceeded_anyway = True
        result = None

harness.check("missing txnId -> enroll_by_aadhaar() is never called (fails fast instead)", not proceeded_anyway)
harness.check("missing txnId -> run() returns early with no enrollment 'response' attempted", result is not None and "response" not in result)



Flow 1: ABHA Enrollment via Aadhaar
      Requesting OTP...

[OK] OTP sent

[FAIL] OTP request returned a 200/success response but no 'txnId' was found in the body -- cannot proceed to submit an OTP without a transaction ID.
PASS -- missing txnId -> enroll_by_aadhaar() is never called (fails fast instead)
PASS -- missing txnId -> run() returns early with no enrollment 'response' attempted


True

---
## M2-4 / M3-4 — several things refreshing the shared ABDM login token at once (confirm-only)

**What these are**: M2-4 ("several things refreshing the shared ABDM login token at once") and M3-4 ("several consent-fetch calls landing at once, right as the login token is about to expire") both describe the exact same underlying race the M1-3 fix above already covers — `get_gateway_token()`'s `_token_cache_lock` serializes EVERY caller within this process, regardless of whether those callers are logins (M1-3's own scenario), HIU consent-fetch callbacks (M3-4), or any other mix of concurrent code paths (M2-4). No additional fix was needed — this cell just confirms the lock generalizes beyond the login-specific scenario it was written for, at a larger concurrency level (15 simultaneous callers instead of 10) to make sure it's not coincidentally login-shaped.

In [10]:
import threading

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None

gen_calls_2 = []
def slow_generate_gateway_token_2():
    time.sleep(0.05)
    idx = len(gen_calls_2)
    gen_calls_2.append(idx)
    return harness.FakeResponse(200, {"accessToken": f"TOKEN_{idx}", "expiresIn": 1800})

with patch("server.auth.generate_gateway_token", slow_generate_gateway_token_2):
    results_2 = []
    def worker_2():
        results_2.append(utils.get_gateway_token())
    # Simulates a realistic mix: some "callers" represent logins (M2-4), others
    # represent independent consent-fetch callbacks (M3-4) -- all just need a
    # valid gateway token at once, right as the cached one is about to expire.
    threads_2 = [threading.Thread(target=worker_2) for _ in range(15)]
    [t.start() for t in threads_2]
    [t.join() for t in threads_2]

harness.check("15 concurrent token-needing callers (logins + consent-fetches) -> exactly 1 real refresh", len(gen_calls_2) == 1)
harness.check("every caller got back the SAME token (no mismatched read mid-refresh)", len(set(results_2)) == 1)

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None


2026-08-15 19:14:47  -> Gateway token refreshed


PASS -- 15 concurrent token-needing callers (logins + consent-fetches) -> exactly 1 real refresh
PASS -- every caller got back the SAME token (no mismatched read mid-refresh)


---
## M1-8 / M1-9 — the certificate-fetch step's own malformed-response handling

**Real-world scenario**: something in front of ABDM (a proxy, load balancer) returns a broken/non-JSON body on the `/profile/public/certificate` fetch specifically (M1-8), or the response IS valid JSON but its `publicKey` field is shaped differently than expected -- e.g. a nested object instead of a plain PEM string (M1-9).

**M1-8 fix**: `crypto.py`'s `download_public_certificate()` already safely parsed the response body ONCE for logging purposes (`response_body`, with a try/except ValueError fallback to `.text`) -- but the actual certificate-parsing logic called `response.json()` a SECOND time, unguarded. Now reuses the already-safely-parsed `response_body` instead, so a non-JSON body raises a clean `ValueError` instead of crashing.

**M1-9 fix**: `publicKey` is now explicitly checked to be a string before `.strip()` is called on it -- a differently-shaped value (dict, list, ...) now raises a clean `ValueError` with a clear message instead of an uncaught `AttributeError`.

In [11]:
from unittest.mock import patch

import server.crypto as crypto

crypto.clear_certificate_cache()
with patch("requests.get", lambda **kw: harness.FakeResponse(200, not_json=True)), \
     patch.object(crypto, "get_gateway_token", lambda: "TOKEN"):
    try:
        crypto.get_public_certificate()
        raised_clean_m1_8 = False
    except ValueError:
        raised_clean_m1_8 = True
    except Exception:
        raised_clean_m1_8 = False
crypto.clear_certificate_cache()

harness.check("M1-8: non-JSON cert response -> a clean ValueError, not an unhandled crash", raised_clean_m1_8)

crypto.clear_certificate_cache()
with patch("requests.get", lambda **kw: harness.FakeResponse(200, {"publicKey": {"nested": "not-a-string"}})), \
     patch.object(crypto, "get_gateway_token", lambda: "TOKEN"):
    try:
        crypto.get_public_certificate()
        raised_clean_m1_9 = False
    except ValueError:
        raised_clean_m1_9 = True
    except Exception:
        raised_clean_m1_9 = False
crypto.clear_certificate_cache()

harness.check("M1-9: non-string publicKey field -> a clean ValueError, not an unhandled crash", raised_clean_m1_9)


2026-08-15 19:15:33     [ERROR] ABDM public certificate response was not a JSON object (got str).
2026-08-15 19:15:33     [ERROR] ABDM public certificate 'publicKey' field was not a string (got dict: {'nested': 'not-a-string'}).


PASS -- M1-8: non-JSON cert response -> a clean ValueError, not an unhandled crash
PASS -- M1-9: non-string publicKey field -> a clean ValueError, not an unhandled crash


True

---
## M1-7 / M1-11 / M1-38 — Flow 8/9's Search ABHA Account step's malformed-response handling

**Real-world scenario**: the `/profile/account/abha/search` call (Flows 8 & 9's shared prerequisite step) returns a broken/non-JSON body (M1-7, mid-flow generically), or a single account object instead of the documented list shape -- the original test plan flagged this as "not 100% sure this is even real ABDM behavior" (M1-11), or the result is missing its `txnId` entirely (M1-38) -- which this flow's own docstring says gets carried through unchanged into the next OTP-request step.

**Fixes** (`tools/m1_test_suite/flows/login_search.py`'s `_search_and_select()`): `response.json()` is now guarded the same way as M1-5's login guards (M1-7); a single dict response is now tolerantly treated as a one-element list instead of crashing on `dict[0]` (M1-11), while any other non-list/non-dict shape is reported as a clean failure; a missing `txnId` now fails fast with a clear message instead of silently carrying `None` through to the next step (M1-38, same shape as M1-15's enrollment guard).

**Note**: `cli.py`'s own top-level dispatch loop already wraps every flow handler in a generic try/except, so none of these three ever fully crashed the CLI process before this fix either -- but they aborted the flow abruptly with a raw exception message instead of this module's normal clear, specific reporting. Also extended the same non-JSON-body guard to `profile_utilities.py`'s two `response.json()` call sites for consistency (those were already caught by that flow's own per-sub-action try/except, so this only improves the message, not the crash-safety).

In [12]:
import tools.m1_test_suite.flows.login_search as login_search

with patch.object(login_search, "search_abha_by_mobile", lambda **kw: harness.FakeResponse(200, not_json=True)), \
     patch.object(login_search, "prompt", lambda *a, **kw: "9999999999"), \
     patch.object(login_search, "encrypt", lambda v: v):
    try:
        result_m1_7 = login_search._search_and_select()
        crashed_m1_7 = False
    except Exception:
        crashed_m1_7 = True
        result_m1_7 = None

harness.check("M1-7: non-JSON search response -> does not crash", not crashed_m1_7)
harness.check("M1-7: treated as a failed search (None, None)", result_m1_7 == (None, None))

with patch.object(login_search, "search_abha_by_mobile", lambda **kw: harness.FakeResponse(200, {"txnId": "txn-1", "ABHA": [{"index": 0, "name": "Test"}]})), \
     patch.object(login_search, "prompt", lambda *a, **kw: "9999999999"), \
     patch.object(login_search, "encrypt", lambda v: v):
    try:
        result_m1_11 = login_search._search_and_select()
        crashed_m1_11 = False
    except Exception:
        crashed_m1_11 = True
        result_m1_11 = None

harness.check("M1-11: a single dict object (not a list) -> does not crash", not crashed_m1_11)
harness.check("M1-11: tolerated as a one-element list, search proceeds normally", result_m1_11 == ("txn-1", 0))

with patch.object(login_search, "search_abha_by_mobile", lambda **kw: harness.FakeResponse(200, [{"ABHA": [{"index": 0, "name": "Test"}]}])), \
     patch.object(login_search, "prompt", lambda *a, **kw: "9999999999"), \
     patch.object(login_search, "encrypt", lambda v: v):
    try:
        result_m1_38 = login_search._search_and_select()
        crashed_m1_38 = False
    except Exception:
        crashed_m1_38 = True
        result_m1_38 = None

harness.check("M1-38: missing txnId -> does not crash", not crashed_m1_38)
harness.check("M1-38: missing txnId -> treated as a failed search, not silently carried through as None", result_m1_38 == (None, None))


      Searching for ABHA accounts linked to this mobile number...

[FAIL] ABHA search returned a 200 status but the response body wasn't valid JSON (No JSON object could be decoded (simulated malformed body)) -- treating this as a failed search rather than crashing.
PASS -- M1-7: non-JSON search response -> does not crash
PASS -- M1-7: treated as a failed search (None, None)
      Searching for ABHA accounts linked to this mobile number...
      Exactly one ABHA account found -- auto-selecting it.
PASS -- M1-11: a single dict object (not a list) -> does not crash
PASS -- M1-11: tolerated as a one-element list, search proceeds normally
      Searching for ABHA accounts linked to this mobile number...

[FAIL] ABHA search returned a 200/success response but no 'txnId' was found in the result -- cannot proceed without a transaction ID to carry through.
PASS -- M1-38: missing txnId -> does not crash
PASS -- M1-38: missing txnId -> treated as a failed search, not silently carried through as 

True